# Modelagem e Avaliação - Dataset SEER Clínico
## Tech Challenge Fase 1 - Saúde e Segurança da Mulher
**FIAP POS TECH - IADT**

**Responsáveis:** Rodrigo (Modelos) + Natalia Cabrera (Integração EDA→Modelagem)

### Objetivo
Treinar e avaliar 5 modelos de classificação para predição de sobrevivência
em pacientes com câncer de mama, utilizando o dataset SEER Clínico.

### Modelos:
1. Regressão Logística
2. Random Forest
3. KNN (K-Nearest Neighbors)
4. Árvore de Decisão
5. XGBoost

### Métrica Principal: **Recall**
Em diagnóstico médico, é crucial minimizar falsos negativos (pacientes em risco
classificados como saudáveis). Por isso, priorizamos Recall.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import sys

sys.path.insert(0, '../../src')
from config import MODELING, FIGURES_DIR, MODELS_DIR
from models import create_models, train_all_models, save_model
from evaluation import (
    evaluate_all_models, plot_confusion_matrix, plot_roc_curves,
    explain_with_shap, get_feature_importance
)

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('Módulos carregados!')

## 1. Carregar Dados Pré-processados

In [ ]:
# Carregar dados do notebook anterior (02_preprocessing)
X_train = pd.read_csv('../../data/processed/seer_X_train.csv')
X_test = pd.read_csv('../../data/processed/seer_X_test.csv')
y_train = pd.read_csv('../../data/processed/seer_y_train.csv').squeeze()
y_test = pd.read_csv('../../data/processed/seer_y_test.csv').squeeze()
feature_names = pd.read_csv('../../data/processed/seer_feature_names.csv').squeeze().tolist()

print(f'Dados carregados:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test: {X_test.shape}')
print(f'  y_train: {y_train.shape} | Distribuição: {y_train.value_counts().to_dict()}')
print(f'  y_test: {y_test.shape} | Distribuição: {y_test.value_counts().to_dict()}')
print(f'  Features: {len(feature_names)}')

## 2. Criar e Treinar Modelos

In [ ]:
# Criar modelos
models = create_models()

# Treinar todos
trained_models = train_all_models(models, X_train, y_train)

## 3. Avaliação Comparativa

In [ ]:
# Avaliar todos os modelos
df_results = evaluate_all_models(trained_models, X_test, y_test)

# Exibir tabela formatada
print('\n\nRanking Final (ordenado por Recall):')
df_results_display = df_results.copy()
for col in df_results_display.columns:
    if df_results_display[col].dtype == float:
        df_results_display[col] = df_results_display[col].apply(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')
df_results_display

## 4. Visualização: Comparação de Métricas

In [ ]:
# Gráfico de barras comparativo
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score']
df_plot = df_results[metrics_to_plot]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(df_plot.index))
width = 0.2
colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3']

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    bars = ax.bar(x + i*width, df_plot[metric], width, label=metric.replace('_', ' ').title(), 
                  color=color, edgecolor='black')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
               f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8, rotation=45)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_plot.index, rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_title('Comparação de Métricas - Dataset SEER Clínico', fontsize=15, fontweight='bold')
ax.legend(loc='lower right')
ax.set_ylim(0.5, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
os.makedirs('../../reports/figures/seer', exist_ok=True)
plt.savefig('../../reports/figures/seer/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Matrizes de Confusão

In [ ]:
# Matriz de confusão para cada modelo
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Óbito', 'Viva'], yticklabels=['Óbito', 'Viva'])
    axes[i].set_title(f'{name}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Predição')
    axes[i].set_ylabel('Real')
    
    # Adicionar métricas
    tn, fp, fn, tp = cm.ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    axes[i].text(0.5, -0.15, f'Recall={recall:.3f} | Specificity={specificity:.3f}',
                transform=axes[i].transAxes, ha='center', fontsize=10)

axes[5].set_visible(False)

plt.suptitle('Matrizes de Confusão - Dataset SEER', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/seer/confusion_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Curvas ROC

In [ ]:
# Curvas ROC de todos os modelos
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Set2(np.linspace(0, 1, len(trained_models)))

for (name, model), color in zip(trained_models.items(), colors):
    try:
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', linewidth=2, color=color)
    except AttributeError:
        print(f'{name}: predict_proba não disponível')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatório (AUC = 0.5)')
ax.set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
ax.set_title('Curvas ROC - Dataset SEER Clínico', fontsize=15, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../reports/figures/seer/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance (Modelos baseados em árvore)

In [ ]:
# Feature importance para Random Forest, XGBoost e Árvore de Decisão
tree_models = ['Random Forest', 'XGBoost', 'Árvore de Decisão']

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for i, name in enumerate(tree_models):
    model = trained_models[name]
    if hasattr(model, 'feature_importances_'):
        imp = model.feature_importances_
        df_imp = pd.DataFrame({'feature': feature_names, 'importance': imp})
        df_imp = df_imp.sort_values('importance', ascending=True).tail(15)
        
        axes[i].barh(df_imp['feature'], df_imp['importance'], color='steelblue', edgecolor='black')
        axes[i].set_title(f'{name}', fontsize=13, fontweight='bold')
        axes[i].set_xlabel('Importância')

plt.suptitle('Feature Importance - Top 15 Features (SEER)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/seer/feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Explicabilidade com SHAP

In [ ]:
# SHAP para o melhor modelo
import shap

best_model_name = df_results.index[0]  # Melhor por recall
best_model = trained_models[best_model_name]
print(f'Gerando SHAP para o melhor modelo: {best_model_name}')

# Escolher explainer adequado
if hasattr(best_model, 'tree_') or hasattr(best_model, 'estimators_') or 'XGB' in best_model.__class__.__name__:
    explainer = shap.TreeExplainer(best_model)
else:
    explainer = shap.LinearExplainer(best_model, X_test)

shap_values = explainer.shap_values(X_test)

# Summary Plot
plt.figure(figsize=(12, 8))
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], X_test, feature_names=feature_names, show=False)
else:
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
plt.title(f'SHAP Summary Plot - {best_model_name} (SEER)', fontsize=14, y=1.02)
plt.subplots_adjust(top=0.92)
plt.savefig('../../reports/figures/seer/shap_summary_best.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot (importância média)
plt.figure(figsize=(12, 8))
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], X_test, feature_names=feature_names, plot_type='bar', show=False)
else:
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance - {best_model_name} (SEER)', fontsize=14, y=1.02)
plt.subplots_adjust(top=0.92)
plt.savefig('../../reports/figures/seer/shap_importance_best.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Salvar Modelos e Resultados

In [ ]:
# Salvar todos os modelos treinados
import joblib
os.makedirs('../../models', exist_ok=True)

for name, model in trained_models.items():
    filepath = f'../../models/seer_{name.replace(" ", "_").replace("á", "a").replace("ã", "a")}.pkl'
    joblib.dump(model, filepath)
    print(f'Salvo: {filepath}')

# Salvar resultados comparativos
df_results.to_csv('../../reports/seer_comparacao_modelos.csv')
print(f'\nResultados salvos em: reports/seer_comparacao_modelos.csv')

print(f'\n{"=" * 60}')
print(f'MELHOR MODELO: {df_results.index[0]}')
print(f'  Recall: {df_results.iloc[0]["recall"]:.4f}')
print(f'  F1-Score: {df_results.iloc[0]["f1_score"]:.4f}')
print(f'  AUC-ROC: {df_results.iloc[0]["auc_roc"]:.4f}')
print(f'{"=" * 60}')

## 10. Discussão e Conclusões

### Análise dos Resultados

**Contexto clínico:** No diagnóstico oncológico, o Recall (sensibilidade) é a métrica
mais crítica. Um falso negativo significa uma paciente em risco que não foi identificada,
podendo atrasar o tratamento e comprometer a sobrevivência.

### Considerações:

1. **Desbalanceamento**: O dataset tem 84.7% de pacientes vivas vs 15.3% de óbitos.
   Usamos `class_weight='balanced'` para compensar.

2. **Feature Engineering**: A criação de Node_Ratio (razão de linfonodos) e
   indicadores de duplo-positivo/negativo hormonal melhoram a capacidade preditiva.

3. **Explicabilidade**: SHAP revela quais fatores clínicos mais influenciam
   a predição, oferecendo transparência crucial em contexto médico.

4. **Aplicação prática**: O modelo pode auxiliar profissionais de saúde na
   triagem e priorização de pacientes, mas **nunca deve substituir** o
   julgamento clínico.

### Próximos Passos:
- Validação cruzada (k-fold) para robustez
- Tuning de hiperparâmetros (GridSearch/RandomSearch)
- Ensemble dos melhores modelos
- Integração com dashboard Power BI (Paula e Thamy)